# 04 · Power BI / Dashboard Dataset Export
Enrich the cleaned data and export CSVs for Power BI or further analysis.

In [ ]:
import pandas as pd, numpy as np, re

df = pd.read_csv("../data/whatsapp_cleaned_data.csv")
df["datetime"] = pd.to_datetime(df["date"]+" "+df["time"], format="%m/%d/%y %I:%M %p", errors="coerce")
df = df.dropna(subset=["datetime"]).sort_values("datetime").reset_index(drop=True)

# Temporal features
df["hour"]      = df["datetime"].dt.hour
df["day"]       = df["datetime"].dt.day_name()
df["month"]     = df["datetime"].dt.month_name()
df["date_only"] = df["datetime"].dt.date

# Text features
df["msg_length"] = df["message"].str.len()
df["word_count"] = df["message"].str.split().str.len()

emoji_re = re.compile("[😀-🙏🌀-🗿🚀-🛿]", re.UNICODE)
df["emoji_count"] = df["message"].apply(lambda x: len(emoji_re.findall(str(x))))

# Response time
df["prev_user"] = df["user"].shift(1)
df["prev_time"] = df["datetime"].shift(1)
df["response_time_min"] = np.where(df["user"]!=df["prev_user"],
    (df["datetime"]-df["prev_time"]).dt.total_seconds()/60, np.nan)
df.drop(columns=["prev_user","prev_time"], inplace=True)
df.head()

In [ ]:
# User summary
user_stats = df.groupby("user").agg(
    total_messages=("message","count"),
    avg_msg_length=("msg_length","mean"),
    avg_word_count=("word_count","mean"),
    total_emojis=("emoji_count","sum")
).reset_index()

# Daily trend
daily = df.groupby("date_only").size().reset_index(name="messages_per_day")

# Save all
df.to_csv("../data/powerbi_chat_dataset.csv",  index=False)
user_stats.to_csv("../data/user_summary.csv",   index=False)
daily.to_csv("../data/daily_trend.csv",          index=False)
print("Exported: powerbi_chat_dataset.csv | user_summary.csv | daily_trend.csv")